# rhythm-core v0.3 RC 실제 곡 검증

태블릿에서도 사용할 수 있는 Colab용 검증 노트북입니다.
1) 프로젝트 RC ZIP 업로드 → 2) 실제 곡 5개 이상 업로드 → 3) 자동 분석 → 4) 결과 ZIP 다운로드 순서입니다.

음원은 결과 ZIP에 포함되지 않습니다.


In [ ]:
from google.colab import files
import os, zipfile, glob, pathlib, json, shutil, subprocess, sys

print("rhythm-core-v0.3-rc ZIP을 업로드하세요.")
uploaded = files.upload()
zip_name = next((n for n in uploaded if n.lower().endswith(".zip")), None)
assert zip_name, "ZIP 파일이 필요합니다."
shutil.rmtree("/content/rhythm-core", ignore_errors=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")
# ZIP 구조가 rhythm-core/ 또는 단일 폴더인 경우 자동 탐색
candidates = [p for p in pathlib.Path("/content").glob("*/analyzer") if (p.parent / "package.json").exists()]
assert candidates, "프로젝트 루트를 찾지 못했습니다."
ROOT = candidates[0].parent
os.chdir(ROOT)
print("project:", ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "analyzer/requirements.txt"] )
print("준비 완료")


In [ ]:
from google.colab import files
import pathlib, shutil

SONGS = pathlib.Path("/content/real-songs")
shutil.rmtree(SONGS, ignore_errors=True)
SONGS.mkdir()
print("실제 테스트곡을 5개 이상 업로드하세요. mp3/wav/flac/ogg/m4a 등을 사용할 수 있습니다.")
uploaded_songs = files.upload()
for name, data in uploaded_songs.items():
    (SONGS / name).write_bytes(data)
print("uploaded:", len(uploaded_songs), "songs")


In [ ]:
import json, pathlib, subprocess, sys

# BPM을 알고 있으면 아래 dict에 파일명: BPM을 적으세요. 모르면 비워둬도 됩니다.
EXPECTED_BPM = {
    # "song.mp3": 174,
}
manifest=[]
for p in sorted(SONGS.iterdir()):
    if not p.is_file():
        continue
    row={"path":str(p), "title":p.stem}
    if p.name in EXPECTED_BPM:
        row["expectedBpm"] = EXPECTED_BPM[p.name]
    manifest.append(row)
assert len(manifest) >= 1
pathlib.Path("/content/real-songs.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
subprocess.check_call([sys.executable, "analyzer/evaluate_real_songs.py", "/content/real-songs.json", "--difficulty", "hard", "--seed", "42", "--out", "/content/evaluation-output"] )
print(pathlib.Path("/content/evaluation-output/summary.csv").read_text(encoding="utf-8"))


## 직접 플레이 후
`evaluation-output/manual-evaluation.csv`의 6개 항목을 1~5점으로 채우면 됩니다.
RC 단계에서는 자동 지표보다 직접 플레이한 **Timing / Musicality / PatternNaturalness / Density / Repetition / OverallFun** 평가가 더 중요합니다.


In [ ]:
from google.colab import files
import shutil
archive = shutil.make_archive("/content/rhythm-v03-evaluation", "zip", "/content/evaluation-output")
files.download(archive)
